In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("luat_dat_dai.pdf")

documents = loader.load()

print(documents[0].page_content[:500])




/home/cyme/tnisl/cs221/demo_11/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VĂN PHÒNG QUỐC HỘI
--------
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT
NAM
Độc lập - Tự do - Hạnh phúc
---------------
Số: 45/VBHN-VPQH
Hà Nội, ngày 28 tháng 02 năm 2025
LUẬT
ĐẤT ĐAI
Luật Đất đai số 31/2024/QH15 ngày 18 tháng 01 năm 2024 của Quốc hội, có hiệu
lực kể từ ngày 01 tháng 8 năm 2024[1], được sửa đổi, bổ sung bởi:
1. Luật số 43/2024/QH15 ngày 29 tháng 6 năm 2024 của Quốc hội sửa đổi, bổ
sung một số điều của Luật Đất đai số 31/2024/QH15, Luật Nhà ở số 27/2023/
QH15, Luật Kinh doanh bất động sản số 


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-m3",
    model_kwargs = {'device': 'cuda'},
    encode_kwargs = {'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 41798.72it/s]


In [ ]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding = embeddings,
    path = "./my_qdrant",
    collection_name = "luat_dat_dai"
)

retriever = vectorstore.as_retriever(search_kwargs = {"k": 2})



In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature=0
)

question = "Điều kiện để hộ gia đình, cá nhân đang sử dụng đất không có giấy tờ được cấp Giấy chứng nhận quyền sử dụng đất (Sổ đỏ) theo quy định mới là gì?"

retrieved_data = retriever.invoke(question)

source = "\n\n---\n\n".join([doc.page_content for doc in retrieved_data])

prompt = f"""


Bạn là một chuyên gia pháp lý thông minh.
Hãy trả lời câu hỏi CHỈ DỰA TRÊN các tài liệu tham khảo dưới đây.
Nếu không có thông tin, hãy nói 'Tài liệu không đề cập'.

TÀI LIỆU THAM KHẢO:
{retrieved_data}

CÂU HỎI CỦA NGƯỜI DÙNG:
{question}

"""

print(prompt)

response = llm.invoke(prompt)

print("\n=== CÂU TRẢ LỜI ===\n")

print(response.content)

print("\n=== NGUỒN ===\n")
for i, doc in enumerate(retrieved_data):
    print(f"--- Nguồn {i+1} ---")
    print(doc.page_content.strip()[:200] + "...\n")

 


Bạn là một chuyên gia pháp lý thông minh. 
Hãy trả lời câu hỏi CHỈ DỰA TRÊN các tài liệu tham khảo dưới đây. 
Nếu không có thông tin, hãy nói 'Tài liệu không đề cập'.

TÀI LIỆU THAM KHẢO:
[Document(metadata={'producer': 'cairo 1.18.0 (https://cairographics.org)', 'creator': 'Mozilla Firefox 149.0', 'creationdate': '2026-04-19T20:49:17+07:00', 'source': 'luat_dat_dai.pdf', 'file_path': 'luat_dat_dai.pdf', 'total_pages': 214, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260419204917+07'00", 'page': 112, '_id': '65a69639141646b9a76f2929e1857cb9', '_collection_name': 'luat_dat_dai'}, page_content='Điều 138. Cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu tài sản\ngắn liền với đất đối với trường hợp hộ gia đình, cá nhân đang sử dụng đất\nkhông có giấy tờ về quyền sử dụng đất mà không vi phạm pháp luật về đất\nđai, không thuộc trường hợp đất được giao không đúng thẩm quyền\nViệc cấp Gi